# Restaurants - WPG Open Data
Mar 5, 2026

In [9]:
import pandas as pd
import time
import requests

In [10]:
df = pd.read_csv("restaurants.csv")

In [11]:
df

,name,latitude,longitude,cuisine
0,Cafe Dario,49.900893,-97.184014,international
1,Casa Ilocandia,49.897472,-97.184361,NaN
2,Bangkok Thai,49.879150,-97.146016,thai
3,Unnamed Restaurant,49.838464,-97.151761,NaN
4,chop Steakhouse and Bar,49.897431,-97.203394,NaN
...,...,...,...,...
511,Pad Thai,49.877082,-97.268230,thai
512,Dreamland Diner,49.876856,-97.267282,diner
513,Spice Enroute,49.969216,-97.131841,indian
514,Olympia Diner,49.881853,-97.292231,diner


In [12]:
df.dtypes.tolist

<bound method IndexOpsMixin.tolist of name          object
latitude     float64
longitude    float64
cuisine       object
dtype: object>

Initial observations:<br>
- missing values in cuisine
- capitalization issues
- Missing restaurant name
- no innate PK
- string "" '' not consistent

Shape;
- 516 rows
- 4 columns
    - 2 float columns
    - 2 object columns

## Deeper Look at Columns:

Get a count of duplicates

In [13]:
counts = df['name'].value_counts()
result = [(name, int(count)) for name, count in zip(counts.index, counts.values)]
result = sorted(result, key=lambda x: x[1], reverse=True)
result

[('Salisbury House', 12),
 ('Unnamed Restaurant', 11),
 ('Boston Pizza', 9),
 ("Smitty's", 8),
 ('Pizza Hut', 7),
 ('Freshii', 7),
 ("Montana's", 4),
 ('Dim Sum Garden', 4),
 ('Santa Lucia Pizza', 3),
 ('Pho Hoang', 3),
 ('Earls', 3),
 ('The Keg', 3),
 ('Pizza Hotline', 3),
 ('Marigold', 2),
 ("Cora's", 2),
 ("Mongo's Grill", 2),
 ("Junior's", 2),
 ('Joey', 2),
 ("Tony Roma's", 2),
 ('Clay Oven', 2),
 ('The Original Pancake House', 2),
 ('Sushi Place', 2),
 ("Stella's", 2),
 ('Aroma Bistro', 2),
 ('Taste of Sri Lanka', 2),
 ('La Pampa Empanadas Gourmet', 2),
 ("Cilantro's", 2),
 ('Perkins', 2),
 ("Original Joe's", 2),
 ('Asia City', 2),
 ('Burrito Splendido', 2),
 ('Cafe 22', 2),
 ('Ben & Florentine', 2),
 ('Olive Garden', 2),
 ('Tom Yum', 2),
 ('Bodegoes', 2),
 ("Moxie's", 2),
 ("Applebee's", 2),
 ('Wok Box', 1),
 ('Market Burger', 1),
 ('Southland Chinese Food Restaurant', 1),
 ('Oriental Pearl', 1),
 ("P.F. Chang's", 1),
 ("InFerno's Bistro", 1),
 ('Jas Indian Cuisine', 1),
 ('The N

11 unnamed resturants. The rest have names.

Find if unamed restuarants all have coordinates attached so they can be filled in.

In [14]:
# Filter unnamed restaurants
unnamed = df[df['name'] == 'Unnamed Restaurant']

# Check their coordinates
print(unnamed[['name', 'latitude', 'longitude']].to_string())

# Summary: how many have/missing coordinates
print("\nMissing coordinates:")
print(unnamed[['latitude', 'longitude']].isnull().sum())

                   name   latitude  longitude
3    Unnamed Restaurant  49.838464 -97.151761
24   Unnamed Restaurant  49.893177 -97.162208
27   Unnamed Restaurant  49.885535 -97.022296
54   Unnamed Restaurant  49.862936 -97.077042
138  Unnamed Restaurant  49.923450 -97.052744
282  Unnamed Restaurant  49.871029 -97.152271
345  Unnamed Restaurant  49.878751 -97.226258
440  Unnamed Restaurant  49.893417 -97.112045
457  Unnamed Restaurant  49.893150 -97.198741
483  Unnamed Restaurant  49.929560 -97.101769
484  Unnamed Restaurant  49.903421 -97.137485

Missing coordinates:
latitude     0
longitude    0
dtype: int64


The unnamed restaurants can be found using coordinates.

This means the cuisine type can also be found.

#### Fill in Restaurant Names

In [15]:
def get_restaurant_name(lat, lon, radius=500):
    query = f"""
    [out:json][timeout:30];
    (
      node(around:{radius},{lat},{lon})[amenity~"restaurant|cafe|fast_food|food_court|bar|pub|bistro"];
      way(around:{radius},{lat},{lon})[amenity~"restaurant|cafe|fast_food|food_court|bar|pub|bistro"];
    );
    out body;
    """
    
    endpoint = "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
    try:
        response = requests.post(endpoint, data=query, timeout=30)
        data = response.json()
        print(f"Found {len(data['elements'])} elements")  # debug
        
        if data['elements']:
            for el in data['elements']:
                name = el.get('tags', {}).get('name')
                if name:
                    return name
        return 'Not found'
    except Exception as e:
        return f'Error: {e}'

test = unnamed.iloc[0]
print(f"Coords: {test['latitude']}, {test['longitude']}")
print(get_restaurant_name(test['latitude'], test['longitude']))

Coords: 49.8384636, -97.1517606
Found 11 elements
Dowon Restaurant


In [16]:
time.sleep(1)

unnamed['suggested_name'] = unnamed.apply(
    lambda row: (time.sleep(1), get_restaurant_name(row['latitude'], row['longitude']))[1],
    axis=1
)

print(unnamed[['latitude', 'longitude', 'suggested_name']])

Found 11 elements
Found 18 elements
Found 1 elements
Found 1 elements
Found 3 elements
Found 34 elements
Found 11 elements
Found 5 elements
Found 14 elements
Found 3 elements
Found 12 elements
      latitude  longitude    suggested_name
3    49.838464 -97.151761  Dowon Restaurant
24   49.893177 -97.162208      Shwarma Time
27   49.885535 -97.022296         Not found
54   49.862936 -97.077042         Not found
138  49.923450 -97.052744      Grand Donuts
282  49.871029 -97.152271           Café 22
345  49.878751 -97.226258         Pizza Hut
440  49.893417 -97.112045  Stella’s au CCFM
457  49.893150 -97.198741           Hooters
483  49.929560 -97.101769          Colosimo
484  49.903421 -97.137485           Sum Hai


C:\Users\QTene\AppData\Local\Temp\ipykernel_111996\4023962564.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unnamed['suggested_name'] = unnamed.apply(


In [17]:
# Replace 'Unnamed Restaurant' with suggested names
df.loc[unnamed.index, 'name'] = unnamed['suggested_name']

# Verify
df.loc[unnamed.index, 'name'].value_counts()

name
Not found           2
Dowon Restaurant    1
Shwarma Time        1
Grand Donuts        1
Café 22             1
Pizza Hut           1
Stella’s au CCFM    1
Hooters             1
Colosimo            1
Sum Hai             1
Name: count, dtype: int64

#### Replace last 2 values

In [18]:
df.loc[unnamed.index, 'name'] # pos 27 and 54 still unnamed

3      Dowon Restaurant
24         Shwarma Time
27            Not found
54            Not found
138        Grand Donuts
282             Café 22
345           Pizza Hut
440    Stella’s au CCFM
457             Hooters
483            Colosimo
484             Sum Hai
Name: name, dtype: object

In [19]:
# display lat and long for pos 27
df.iloc[27]

name         Not found
latitude     49.885535
longitude   -97.022296
cuisine            NaN
Name: 27, dtype: object

Not a real restuarant. Private venue that accepts catering. Will remove from df

In [20]:
df.drop(index=27)

,name,latitude,longitude,cuisine
0,Cafe Dario,49.900893,-97.184014,international
1,Casa Ilocandia,49.897472,-97.184361,NaN
2,Bangkok Thai,49.879150,-97.146016,thai
3,Dowon Restaurant,49.838464,-97.151761,NaN
4,chop Steakhouse and Bar,49.897431,-97.203394,NaN
...,...,...,...,...
511,Pad Thai,49.877082,-97.268230,thai
512,Dreamland Diner,49.876856,-97.267282,diner
513,Spice Enroute,49.969216,-97.131841,indian
514,Olympia Diner,49.881853,-97.292231,diner


In [21]:
# display lat and long for pos 54
df.iloc[54]

name         Not found
latitude     49.862936
longitude   -97.077042
cuisine            NaN
Name: 54, dtype: object

Restaurant is Komodo Chinese Restaurant

In [22]:
df['name'] = df['name'].replace('Not found', 'Komodo Chinese Restaurant')

In [23]:
row = df[df['name'] == 'Komodo Chinese Restaurant']
row

,name,latitude,longitude,cuisine
27,Komodo Chinese Restaurant,49.885535,-97.022296,NaN
54,Komodo Chinese Restaurant,49.862936,-97.077042,NaN


### Summary of this dataset:

Possible uses cases:
- Map of restuarants
    - colour code map points by cuisine
- Training data for LLM
- Itinerary builder (proximity of restaurants)

Other Data Needed:
- Price range
- hours
- happy hours
- ratings
- local or not

#### Cleaning done:

- removed row 27 (not a restaurant)
- replaced other rows with restaurant names
- replaced 'not found' in row 54 with restaurant name